# 06 - Batch Reprocessing (Lambda Architecture)
Retraitement periodique des donnees streaming accumulees dans HDFS

**Principe** : les donnees arrivent en streaming (notebook 04/05) et sont stockees dans HDFS.
A intervalle regulier (configurable), ce notebook lance un traitement batch complet
sur toutes les donnees accumulees pour produire des resultats plus precis.

```
Streaming (temps reel)          Batch Reprocessing (periodique)
   ├─ approximatif               ├─ exact et complet
   ├─ latence faible             ├─ latence = intervalle
   └─ ClickHouse stream_*        └─ ClickHouse reprocess_*
```

In [1]:
import subprocess
subprocess.check_call(['pip', 'install', 'clickhouse-connect'])

import builtins
import time
import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import clickhouse_connect

spark = SparkSession.builder \
    .appName('06_Batch_Reprocessing') \
    .master('spark://spark-master:7077') \
    .config('spark.hadoop.fs.defaultFS', 'hdfs://namenode:9000') \
    .config('spark.sql.shuffle.partitions', '6') \
    .getOrCreate()

ch_client = clickhouse_connect.get_client(host='clickhouse', port=8123)

# Charger les dimensions (statiques)
locations = spark.read.parquet('hdfs://namenode:9000/projet/silver/locations_clean/')
dates = spark.read.parquet('hdfs://namenode:9000/projet/silver/dates_clean/')

print('Spark connecte')
print('ClickHouse connecte')
print(f'Locations: {locations.count()} zones')

Spark connecte
ClickHouse connecte
Locations: 263 zones


In [2]:
# ══════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════

HDFS_STREAMING_PATH = 'hdfs://namenode:9000/projet/streaming/december_trips'

# Intervalle entre chaque batch reprocessing (en secondes)
# Exemples :
#   30   = toutes les 30 secondes (pour la demo)
#   300  = toutes les 5 minutes
#   3600 = toutes les heures
BATCH_INTERVAL_SECONDS = 30

# Nombre de cycles de reprocessing a executer
# Mettre -1 pour tourner indefiniment (arreter manuellement)
NB_CYCLES = 5

print(f'Intervalle : {BATCH_INTERVAL_SECONDS} secondes')
print(f'Cycles prevus : {NB_CYCLES}')
print(f'Source HDFS : {HDFS_STREAMING_PATH}')

Intervalle : 30 secondes
Cycles prevus : 5
Source HDFS : hdfs://namenode:9000/projet/streaming/december_trips


In [3]:
# ══════════════════════════════════════════════════
# FONCTION DE BATCH REPROCESSING
# ══════════════════════════════════════════════════
# Lit TOUTES les donnees accumulees dans HDFS et
# recalcule les agregations de zero (plus precis
# que les resultats streaming incrementaux)

def run_batch_reprocessing(cycle_num):
    """Execute un cycle de batch reprocessing sur les donnees streaming accumulees"""
    
    start = datetime.datetime.now()
    print(f'\n{"="*60}')
    print(f'CYCLE {cycle_num} - {start.strftime("%H:%M:%S")}')
    print(f'{"="*60}')
    
    # ── Lire les donnees accumulees depuis HDFS ──
    try:
        trips = spark.read.parquet(HDFS_STREAMING_PATH)
        nb_rows = trips.count()
        print(f'Donnees accumulees dans HDFS : {nb_rows} lignes')
    except Exception as e:
        print(f'Pas encore de donnees dans HDFS : {e}')
        return
    
    if nb_rows == 0:
        print('Aucune donnee a traiter')
        return
    
    # ── Agregation 1 : Stats par borough (batch exact) ──
    stats_borough = (trips
        .join(broadcast(locations),
              trips.pickup_location_id == locations.location_id, 'left')
        .groupBy('borough')
        .agg(
            count('*').alias('nb_trips'),
            round(sum('total_amount'), 2).alias('total_revenue'),
            round(avg('trip_distance'), 2).alias('avg_distance'),
            round(avg('total_amount'), 2).alias('avg_fare')
        )
        .orderBy(desc('nb_trips'))
    )
    
    print('\n--- Stats Borough (batch reprocess) ---')
    stats_borough.show()
    
    # Ecrire dans ClickHouse (ecrase les resultats precedents)
    pdf_borough = stats_borough.toPandas()
    pdf_borough = pdf_borough.dropna(subset=['borough'])
    ch_client.command('TRUNCATE TABLE reprocess_borough_stats')
    ch_client.insert_df('reprocess_borough_stats', pdf_borough)
    
    # ── Agregation 2 : Top zones (batch exact) ──
    top_zones = (trips
        .join(broadcast(locations),
              trips.pickup_location_id == locations.location_id, 'left')
        .groupBy('zone', 'borough')
        .agg(count('*').alias('nb_trips'))
        .orderBy(desc('nb_trips'))
        .limit(10)
    )
    
    print('--- Top 10 Zones (batch reprocess) ---')
    top_zones.show(truncate=False)
    
    pdf_zones = top_zones.toPandas()
    pdf_zones = pdf_zones.dropna(subset=['zone'])
    ch_client.command('TRUNCATE TABLE reprocess_top_zones')
    ch_client.insert_df('reprocess_top_zones', pdf_zones)
    
    # ── Agregation 3 : Stats par jour (batch exact) ──
    stats_daily = (trips
        .join(broadcast(dates), trips.date_id == dates.date_id, 'left')
        .groupBy('day')
        .agg(
            count('*').alias('nb_trips'),
            round(sum('total_amount'), 2).alias('total_revenue'),
            round(avg('total_amount'), 2).alias('avg_fare')
        )
        .orderBy('day')
    )
    
    print('--- Stats par Jour de Decembre (batch reprocess) ---')
    stats_daily.show()
    
    pdf_daily = stats_daily.toPandas()
    pdf_daily = pdf_daily.dropna(subset=['day'])
    ch_client.command('TRUNCATE TABLE reprocess_daily_stats')
    ch_client.insert_df('reprocess_daily_stats', pdf_daily)
    
    # ── Log du cycle ──
    end = datetime.datetime.now()
    duration = (end - start).total_seconds()
    
    log_entry = {
        'cycle_num': cycle_num,
        'executed_at': end.strftime('%Y-%m-%d %H:%M:%S'),
        'rows_processed': nb_rows,
        'duration_seconds': builtins.round(duration, 1),
        'boroughs_found': len(pdf_borough),
        'zones_found': len(pdf_zones)
    }
    import pandas as pd
    ch_client.insert_df('reprocess_log', pd.DataFrame([log_entry]))
    
    print(f'\nCycle {cycle_num} termine en {builtins.round(duration, 1)}s')
    print(f'  {nb_rows} lignes traitees → ClickHouse reprocess_*')

In [4]:
# ══════════════════════════════════════════════════
# BOUCLE DE REPROCESSING PERIODIQUE
# ══════════════════════════════════════════════════
# Lance un batch toutes les BATCH_INTERVAL_SECONDS
# A chaque cycle, les donnees accumulees dans HDFS
# grandissent (le streaming continue d'ecrire)
# → le batch reprocesse TOUT depuis le debut

print(f'Demarrage du batch reprocessing periodique')
print(f'Intervalle : toutes les {BATCH_INTERVAL_SECONDS} secondes')
print(f'Cycles : {NB_CYCLES}')
print(f'\nLancez le producer (notebook 03) et le consumer HDFS (notebook 05)')
print(f'en parallele pour voir les donnees augmenter a chaque cycle.\n')

cycle = 1
while NB_CYCLES == -1 or cycle <= NB_CYCLES:
    run_batch_reprocessing(cycle)
    
    if NB_CYCLES != -1 and cycle >= NB_CYCLES:
        break
    
    print(f'\nProchain batch dans {BATCH_INTERVAL_SECONDS} secondes...')
    time.sleep(BATCH_INTERVAL_SECONDS)
    cycle += 1

print(f'\n{"="*60}')
print(f'BATCH REPROCESSING TERMINE - {cycle} cycles executes')
print(f'{"="*60}')

Demarrage du batch reprocessing periodique
Intervalle : toutes les 30 secondes
Cycles : 5

Lancez le producer (notebook 03) et le consumer HDFS (notebook 05)
en parallele pour voir les donnees augmenter a chaque cycle.


CYCLE 1 - 02:50:00
Donnees accumulees dans HDFS : 53675 lignes

--- Stats Borough (batch reprocess) ---
+-------------+--------+-------------+------------+--------+
|      borough|nb_trips|total_revenue|avg_distance|avg_fare|
+-------------+--------+-------------+------------+--------+
|    Manhattan|   26671|    672675.17|        2.41|   25.22|
|        Bronx|   11885|    380067.54|        3.34|   31.98|
|       Queens|    8821|     357223.1|        5.97|    40.5|
|     Brooklyn|    5678|    154432.26|        2.89|    27.2|
|Staten Island|     603|     17859.43|        3.14|   29.62|
|          EWR|      17|      1669.88|        0.01|   98.23|
+-------------+--------+-------------+------------+--------+



DatabaseError: Received ClickHouse exception, code: 60, server response: Code: 60. DB::Exception: Table default.reprocess_borough_stats does not exist. Maybe you meant stream_borough_stats?. (UNKNOWN_TABLE) (version 23.8.16.16 (official build)) (for url http://clickhouse:8123)

In [ ]:
# ══════════════════════════════════════════════════
# COMPARAISON : Streaming vs Batch Reprocess
# ══════════════════════════════════════════════════
# Montre la difference entre les resultats streaming
# (approximatifs, incrementaux) et les resultats batch
# (exacts, recalcules de zero)

print('=== STREAMING (temps reel, approximatif) ===')
df_stream = ch_client.query_df('SELECT * FROM stream_borough_stats FINAL ORDER BY nb_trips DESC')
print(df_stream)

print('\n=== BATCH REPROCESS (periodique, exact) ===')
df_batch = ch_client.query_df('SELECT * FROM reprocess_borough_stats ORDER BY nb_trips DESC')
print(df_batch)

print('\n=== HISTORIQUE DES CYCLES ===')
df_log = ch_client.query_df('SELECT * FROM reprocess_log ORDER BY cycle_num')
print(df_log)

In [ ]:
ch_client.close()
spark.stop()
print('Connexions fermees')